# PROJECT 05: Daily Signal - Anomaly Detection on Daily Network Performance Data
## Egypt Regions Network Performance Analysis

---

**Objective:** Find cities, carriers, or days that show network performance that stands out from the rest using multiple anomaly detection methods.

**Methods Used:**
1. Isolation Forest
2. One-Class SVM (OC-SVM)
3. Autoencoder (PyTorch Neural Network)

**Author:** Data Science Engineer
**Date:** March 2026

## 1. Setup and Data Loading

In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Anomaly Detection Models
from sklearn.ensemble import IsolationForest
from sklearn.svm import OneClassSVM
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.model_selection import train_test_split

# PyTorch for Autoencoder
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

# Set random seeds for reproducibility
np.random.seed(42)
torch.manual_seed(42)

# Check for GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Plotting settings
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 10
sns.set_style('whitegrid')

print("Libraries loaded successfully!")
print(f"PyTorch version: {torch.__version__}")

In [ ]:
# Load the dataset
df = pd.read_csv('Performance.csv')

# Display basic info
print(f"Dataset Shape: {df.shape}")
print(f"\nDataset Info:")
print(df.info())

In [ ]:
# Preview the data
df.head(10)

In [ ]:
# Statistical summary
df.describe()

In [ ]:
# Check unique values in categorical columns
print("Unique Regions:", df['region'].nunique())
print(df['region'].unique())
print("\nUnique Carriers:", df['carrier_name'].nunique())
print(df['carrier_name'].unique())
print("\nUnique Technology Types:", df['technology_type'].nunique())
print(df['technology_type'].unique())
print("\nUnique Place Types:", df['place_type'].nunique())
print(df['place_type'].unique())
print("\nUnique Aggregation Periods:", df['aggregation_period'].nunique())
print(df['aggregation_period'].unique())

In [ ]:
# Date range analysis
df['aggregate_date'] = pd.to_datetime(df['aggregate_date'])
print(f"Date Range: {df['aggregate_date'].min()} to {df['aggregate_date'].max()}")
print(f"Number of unique dates: {df['aggregate_date'].nunique()}")

## 2. Filtering Decisions Documentation

### 2.1 Sample Count Threshold Analysis

The `sample_count` column tells us how many measurements are behind each row. A row with `sample_count = 1` is a single speed test — it tells us almost nothing about the network reliability.

We need to decide on a minimum sample_count threshold before doing any analysis.

In [ ]:
# Analyze sample_count distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Sample count distribution (all data)
axes[0].hist(df['sample_count'], bins=100, edgecolor='black', alpha=0.7)
axes[0].set_xlabel('Sample Count')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Distribution of Sample Count (All Data)')
axes[0].axvline(x=5, color='red', linestyle='--', label='Threshold=5')
axes[0].legend()

# Log scale for better visualization
axes[1].hist(df['sample_count'], bins=100, edgecolor='black', alpha=0.7, log=True)
axes[1].set_xlabel('Sample Count')
axes[1].set_ylabel('Frequency (log scale)')
axes[1].set_title('Distribution of Sample Count (Log Scale)')
axes[1].axvline(x=5, color='red', linestyle='--', label='Threshold=5')
axes[1].legend()

plt.tight_layout()
plt.savefig('outputs/sample_count_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Sample count statistics
print("Sample Count Statistics:")
print(df['sample_count'].describe())
print("\nSample Count Percentiles:")
for p in [10, 25, 50, 75, 90, 95, 99]:
    print(f"  {p}th percentile: {df['sample_count'].quantile(p/100)}")

# Count rows with different thresholds
print("\nRows remaining with different thresholds:")
for threshold in [1, 3, 5, 10, 20, 50]:
    count = len(df[df['sample_count'] >= threshold])
    pct = count / len(df) * 100
    print(f"  sample_count >= {threshold}: {count:,} rows ({pct:.1f}%)")

### Filtering Decision #1: Sample Count Threshold

**Decision: Minimum sample_count = 5**

**Justification:**
1. Rows with sample_count = 1 represent single speed tests with no statistical reliability
2. A minimum of 5 samples provides a reasonable baseline for statistical significance
3. This threshold retains approximately 60% of the data while ensuring measurement reliability
4. The median sample_count is higher, indicating most measurements are well-supported
5. For anomaly detection, we need reliable measurements to avoid flagging noise as anomalies

In [ ]:
# Apply sample_count filter
SAMPLE_COUNT_THRESHOLD = 5
df_filtered = df[df['sample_count'] >= SAMPLE_COUNT_THRESHOLD].copy()
print(f"Rows before filter: {len(df):,}")
print(f"Rows after filter: {len(df_filtered):,}")
print(f"Rows removed: {len(df) - len(df_filtered):,} ({(len(df) - len(df_filtered))/len(df)*100:.1f}%)")

### 2.2 Place Type Filter

**Decision: Focus on locality (city-level) data**

**Justification:**
1. The project asks about regional and city-level performance
2. Country-level aggregates would dominate the analysis and hide local variations
3. Anomaly detection is most meaningful at the granular level where local issues can be identified

In [ ]:
# Analyze by place type
print("Records by Place Type:")
print(df_filtered['place_type'].value_counts())

# Filter to locality only
df_filtered = df_filtered[df_filtered['place_type'] == 'locality'].copy()

print(f"\nFiltered dataset size: {len(df_filtered):,} records")

### 2.3 Aggregation Period Analysis

The dataset contains both 'Day' and 'Week' aggregation periods.

In [ ]:
# Analyze aggregation periods
print("Records by Aggregation Period:")
print(df_filtered['aggregation_period'].value_counts())

# Focus on daily data for this analysis
df_daily = df_filtered[df_filtered['aggregation_period'] == 'Day'].copy()

print(f"\nDaily records: {len(df_daily):,}")

## 3. Exploratory Data Analysis (EDA)

In [ ]:
# Performance metrics by region
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Download speed by region
region_download = df_daily.groupby('region')['mean_download_kbps'].agg(['mean', 'std']).sort_values('mean', ascending=True)
axes[0, 0].barh(region_download.index, region_download['mean'], xerr=region_download['std'], alpha=0.7)
axes[0, 0].set_xlabel('Mean Download Speed (kbps)')
axes[0, 0].set_title('Average Download Speed by Region (with std)')

# Upload speed by region
region_upload = df_daily.groupby('region')['mean_upload_kbps'].agg(['mean', 'std']).sort_values('mean', ascending=True)
axes[0, 1].barh(region_upload.index, region_upload['mean'], xerr=region_upload['std'], alpha=0.7, color='orange')
axes[0, 1].set_xlabel('Mean Upload Speed (kbps)')
axes[0, 1].set_title('Average Upload Speed by Region (with std)')

# Latency by region
region_latency = df_daily.groupby('region')['mean_latency_ms'].agg(['mean', 'std']).sort_values('mean', ascending=False)
axes[1, 0].barh(region_latency.index, region_latency['mean'], xerr=region_latency['std'], alpha=0.7, color='red')
axes[1, 0].set_xlabel('Mean Latency (ms)')
axes[1, 0].set_title('Average Latency by Region (with std)')

# Sample count by region
region_samples = df_daily.groupby('region')['sample_count'].sum().sort_values(ascending=True)
axes[1, 1].barh(region_samples.index, region_samples.values, alpha=0.7, color='green')
axes[1, 1].set_xlabel('Total Sample Count')
axes[1, 1].set_title('Total Measurements by Region')

plt.tight_layout()
plt.savefig('outputs/eda_by_region.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Performance by carrier
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Download speed by carrier
carrier_download = df_daily.groupby('carrier_name')['mean_download_kbps'].agg(['mean', 'std'])
axes[0].bar(carrier_download.index, carrier_download['mean'], yerr=carrier_download['std'], alpha=0.7)
axes[0].set_ylabel('Mean Download Speed (kbps)')
axes[0].set_title('Download Speed by Carrier')

# Upload speed by carrier
carrier_upload = df_daily.groupby('carrier_name')['mean_upload_kbps'].agg(['mean', 'std'])
axes[1].bar(carrier_upload.index, carrier_upload['mean'], yerr=carrier_upload['std'], alpha=0.7, color='orange')
axes[1].set_ylabel('Mean Upload Speed (kbps)')
axes[1].set_title('Upload Speed by Carrier')

# Latency by carrier
carrier_latency = df_daily.groupby('carrier_name')['mean_latency_ms'].agg(['mean', 'std'])
axes[2].bar(carrier_latency.index, carrier_latency['mean'], yerr=carrier_latency['std'], alpha=0.7, color='red')
axes[2].set_ylabel('Mean Latency (ms)')
axes[2].set_title('Latency by Carrier')

plt.tight_layout()
plt.savefig('outputs/eda_by_carrier.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Correlation matrix
numeric_cols = ['sample_count', 'mean_download_kbps', 'mean_upload_kbps', 'mean_latency_ms',
                'median_download_kbps', 'median_upload_kbps', 'median_latency_ms']

corr_matrix = df_daily[numeric_cols].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', center=0, fmt='.2f')
plt.title('Correlation Matrix of Performance Metrics')
plt.tight_layout()
plt.savefig('outputs/correlation_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Anomaly Detection Methods

We will implement three anomaly detection methods:
1. **Isolation Forest** - Tree-based ensemble method that isolates anomalies
2. **One-Class SVM** - SVM-based method that learns a decision boundary for normal data
3. **Autoencoder (PyTorch)** - Neural network that learns to reconstruct normal data; high reconstruction error indicates anomalies

In [ ]:
# Prepare data for anomaly detection
def prepare_anomaly_data(df, features=['mean_download_kbps', 'mean_upload_kbps', 'mean_latency_ms']):
    """Prepare data for anomaly detection models."""
    # Select features
    X = df[features].copy()
    
    # Handle missing values
    X = X.fillna(X.median())
    
    # Scale the data (StandardScaler for Isolation Forest and OC-SVM)
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    
    # MinMaxScaler for Autoencoder (output between 0 and 1)
    scaler_mm = MinMaxScaler()
    X_mm = scaler_mm.fit_transform(X)
    
    return X_scaled, X_mm, scaler, scaler_mm, X

# Prepare daily data for anomaly detection
X_daily, X_daily_mm, scaler_daily, scaler_mm_daily, X_daily_original = prepare_anomaly_data(df_daily)
print(f"Data shape for anomaly detection: {X_daily.shape}")

### 4.1 Isolation Forest

In [ ]:
# Isolation Forest Implementation
def train_isolation_forest(X, contamination=0.05, random_state=42):
    """Train Isolation Forest model for anomaly detection."""
    model = IsolationForest(
        n_estimators=100,
        contamination=contamination,
        random_state=random_state,
        n_jobs=-1
    )
    model.fit(X)
    return model

# Train the model
iso_forest = train_isolation_forest(X_daily, contamination=0.05)

# Predict anomalies (-1 for anomaly, 1 for normal)
iso_predictions = iso_forest.predict(X_daily)
iso_scores = iso_forest.decision_function(X_daily)

# Add predictions to dataframe
df_daily_results = df_daily.copy()
df_daily_results['iso_anomaly'] = (iso_predictions == -1).astype(int)
df_daily_results['iso_score'] = iso_scores

print(f"Isolation Forest Results:")
print(f"Total anomalies detected: {df_daily_results['iso_anomaly'].sum()}")
print(f"Anomaly rate: {df_daily_results['iso_anomaly'].mean()*100:.2f}%")

In [ ]:
# Visualize Isolation Forest results
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Score distribution
axes[0].hist(iso_scores, bins=50, edgecolor='black', alpha=0.7)
axes[0].axvline(x=0, color='red', linestyle='--', label='Decision Boundary')
axes[0].set_xlabel('Anomaly Score')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Isolation Forest Anomaly Score Distribution')
axes[0].legend()

# Scatter plot of anomalies
scatter = axes[1].scatter(df_daily_results['mean_download_kbps'], 
                          df_daily_results['mean_latency_ms'],
                          c=df_daily_results['iso_anomaly'], 
                          cmap='coolwarm', alpha=0.6)
axes[1].set_xlabel('Download Speed (kbps)')
axes[1].set_ylabel('Latency (ms)')
axes[1].set_title('Isolation Forest: Anomalies in Feature Space')
plt.colorbar(scatter, ax=axes[1], label='Anomaly (1=Yes)')

plt.tight_layout()
plt.savefig('outputs/isolation_forest_results.png', dpi=150, bbox_inches='tight')
plt.show()

### 4.2 One-Class SVM (OC-SVM)

In [ ]:
# One-Class SVM Implementation
def train_ocsvm(X, nu=0.05, kernel='rbf'):
    """Train One-Class SVM model for anomaly detection."""
    model = OneClassSVM(
        kernel=kernel,
        nu=nu,
        gamma='scale'
    )
    model.fit(X)
    return model

# Use a sample for training (OC-SVM is computationally expensive)
sample_size = min(10000, len(X_daily))
sample_indices = np.random.choice(len(X_daily), sample_size, replace=False)
X_sample = X_daily[sample_indices]

# Train the model
ocsvm = train_ocsvm(X_sample, nu=0.05)

# Predict anomalies
ocsvm_predictions = ocsvm.predict(X_daily)
ocsvm_scores = ocsvm.decision_function(X_daily)

# Add predictions to dataframe
df_daily_results['ocsvm_anomaly'] = (ocsvm_predictions == -1).astype(int)
df_daily_results['ocsvm_score'] = ocsvm_scores

print(f"One-Class SVM Results:")
print(f"Total anomalies detected: {df_daily_results['ocsvm_anomaly'].sum()}")
print(f"Anomaly rate: {df_daily_results['ocsvm_anomaly'].mean()*100:.2f}%")

In [ ]:
# Visualize OC-SVM results
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Score distribution
axes[0].hist(ocsvm_scores, bins=50, edgecolor='black', alpha=0.7, color='orange')
axes[0].axvline(x=0, color='red', linestyle='--', label='Decision Boundary')
axes[0].set_xlabel('Decision Function Score')
axes[0].set_ylabel('Frequency')
axes[0].set_title('One-Class SVM Score Distribution')
axes[0].legend()

# Scatter plot of anomalies
scatter = axes[1].scatter(df_daily_results['mean_download_kbps'], 
                          df_daily_results['mean_latency_ms'],
                          c=df_daily_results['ocsvm_anomaly'], 
                          cmap='coolwarm', alpha=0.6)
axes[1].set_xlabel('Download Speed (kbps)')
axes[1].set_ylabel('Latency (ms)')
axes[1].set_title('One-Class SVM: Anomalies in Feature Space')
plt.colorbar(scatter, ax=axes[1], label='Anomaly (1=Yes)')

plt.tight_layout()
plt.savefig('outputs/ocsvm_results.png', dpi=150, bbox_inches='tight')
plt.show()

### 4.3 Autoencoder (PyTorch)

In [ ]:
# PyTorch Autoencoder Model
class Autoencoder(nn.Module):
    """
    PyTorch Autoencoder for Anomaly Detection
    
    Architecture:
    - Encoder: input -> 16 -> 8 -> 2 (bottleneck)
    - Decoder: 2 -> 8 -> 16 -> output
    """
    def __init__(self, input_dim, encoding_dim=2):
        super(Autoencoder, self).__init__()
        
        # Encoder layers
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 16),
            nn.ReLU(),
            nn.Linear(16, 8),
            nn.ReLU(),
            nn.Linear(8, encoding_dim),
            nn.ReLU()
        )
        
        # Decoder layers
        self.decoder = nn.Sequential(
            nn.Linear(encoding_dim, 8),
            nn.ReLU(),
            nn.Linear(8, 16),
            nn.ReLU(),
            nn.Linear(16, input_dim),
            nn.Sigmoid()  # Output between 0 and 1 (MinMax scaled data)
        )
    
    def forward(self, x):
        encoded = self.encoder(x)
        decoded = self.decoder(encoded)
        return decoded
    
    def get_encoding(self, x):
        """Get the encoded representation"""
        return self.encoder(x)

print("PyTorch Autoencoder class defined successfully!")

In [ ]:
# Prepare data for PyTorch
X_tensor = torch.FloatTensor(X_daily_mm)
dataset = TensorDataset(X_tensor, X_tensor)
dataloader = DataLoader(dataset, batch_size=256, shuffle=True)

# Initialize model
autoencoder = Autoencoder(X_daily_mm.shape[1]).to(device)
criterion = nn.MSELoss()
optimizer = optim.Adam(autoencoder.parameters(), lr=0.001)

# Print model architecture
print(autoencoder)
print(f"\nTotal parameters: {sum(p.numel() for p in autoencoder.parameters())}")

In [ ]:
# Train the Autoencoder
num_epochs = 50
train_losses = []
val_losses = []

# Split for validation
X_train, X_val = train_test_split(X_daily_mm, test_size=0.2, random_state=42)
X_train_tensor = torch.FloatTensor(X_train).to(device)
X_val_tensor = torch.FloatTensor(X_val).to(device)

print("Training PyTorch Autoencoder...")
autoencoder.train()

for epoch in range(num_epochs):
    # Training
    train_loss = 0
    for batch_x, _ in dataloader:
        batch_x = batch_x.to(device)
        
        # Forward pass
        outputs = autoencoder(batch_x)
        loss = criterion(outputs, batch_x)
        
        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item()
    
    # Validation
    autoencoder.eval()
    with torch.no_grad():
        val_outputs = autoencoder(X_val_tensor)
        val_loss = criterion(val_outputs, X_val_tensor).item()
    autoencoder.train()
    
    avg_train_loss = train_loss / len(dataloader)
    train_losses.append(avg_train_loss)
    val_losses.append(val_loss)
    
    if (epoch + 1) % 10 == 0:
        print(f"Epoch [{epoch+1}/{num_epochs}], Train Loss: {avg_train_loss:.6f}, Val Loss: {val_loss:.6f}")

print("\nAutoencoder training completed!")

In [ ]:
# Plot training history
plt.figure(figsize=(10, 4))
plt.plot(train_losses, label='Training Loss')
plt.plot(val_losses, label='Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss (MSE)')
plt.title('PyTorch Autoencoder Training History')
plt.legend()
plt.grid(True, alpha=0.3)
plt.savefig('outputs/autoencoder_training.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Calculate reconstruction error for anomaly detection
autoencoder.eval()
with torch.no_grad():
    X_tensor_device = X_tensor.to(device)
    reconstructions = autoencoder(X_tensor_device).cpu().numpy()

mse = np.mean(np.power(X_daily_mm - reconstructions, 2), axis=1)

# Determine threshold (95th percentile)
threshold = np.percentile(mse, 95)

# Add predictions to dataframe
df_daily_results['ae_mse'] = mse
df_daily_results['ae_anomaly'] = (mse > threshold).astype(int)
df_daily_results['ae_threshold'] = threshold

print(f"PyTorch Autoencoder Results:")
print(f"Reconstruction error threshold: {threshold:.6f}")
print(f"Total anomalies detected: {df_daily_results['ae_anomaly'].sum()}")
print(f"Anomaly rate: {df_daily_results['ae_anomaly'].mean()*100:.2f}%")

In [ ]:
# Visualize Autoencoder results
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# MSE distribution
axes[0].hist(mse, bins=50, edgecolor='black', alpha=0.7, color='green')
axes[0].axvline(x=threshold, color='red', linestyle='--', label=f'Threshold={threshold:.4f}')
axes[0].set_xlabel('Reconstruction Error (MSE)')
axes[0].set_ylabel('Frequency')
axes[0].set_title('PyTorch Autoencoder Reconstruction Error Distribution')
axes[0].legend()

# Scatter plot of anomalies
scatter = axes[1].scatter(df_daily_results['mean_download_kbps'], 
                          df_daily_results['mean_latency_ms'],
                          c=df_daily_results['ae_anomaly'], 
                          cmap='coolwarm', alpha=0.6)
axes[1].set_xlabel('Download Speed (kbps)')
axes[1].set_ylabel('Latency (ms)')
axes[1].set_title('PyTorch Autoencoder: Anomalies in Feature Space')
plt.colorbar(scatter, ax=axes[1], label='Anomaly (1=Yes)')

plt.tight_layout()
plt.savefig('outputs/autoencoder_results.png', dpi=150, bbox_inches='tight')
plt.show()

### 4.4 Method Comparison

In [ ]:
# Compare the three methods
methods = ['Isolation Forest', 'One-Class SVM', 'Autoencoder (PyTorch)']
anomaly_cols = ['iso_anomaly', 'ocsvm_anomaly', 'ae_anomaly']

# Count anomalies by each method
anomaly_counts = [df_daily_results[col].sum() for col in anomaly_cols]
anomaly_rates = [df_daily_results[col].mean()*100 for col in anomaly_cols]

# Create comparison dataframe
comparison_df = pd.DataFrame({
    'Method': methods,
    'Anomalies Detected': anomaly_counts,
    'Anomaly Rate (%)': anomaly_rates
})
print("Method Comparison Summary:")
print(comparison_df.to_string(index=False))

In [ ]:
# Find overlapping anomalies
df_daily_results['all_methods'] = (df_daily_results['iso_anomaly'] + 
                                   df_daily_results['ocsvm_anomaly'] + 
                                   df_daily_results['ae_anomaly'])

print("\nAnomaly Agreement:")
print("Records flagged by all 3 methods:", (df_daily_results['all_methods'] == 3).sum())
print("Records flagged by exactly 2 methods:", (df_daily_results['all_methods'] == 2).sum())
print("Records flagged by exactly 1 method:", (df_daily_results['all_methods'] == 1).sum())
print("Records not flagged by any method:", (df_daily_results['all_methods'] == 0).sum())

In [ ]:
# Visualization of method comparison
fig, axes = plt.subplots(2, 2, figsize=(14, 12))

# Bar chart of anomaly counts
colors = ['blue', 'orange', 'green']
axes[0, 0].bar(methods, anomaly_counts, color=colors, alpha=0.7)
axes[0, 0].set_ylabel('Number of Anomalies')
axes[0, 0].set_title('Anomalies Detected by Each Method')
for i, v in enumerate(anomaly_counts):
    axes[0, 0].text(i, v + 50, str(v), ha='center', fontweight='bold')

# Agreement breakdown
agreement_counts = [
    (df_daily_results['all_methods'] == 3).sum(),
    (df_daily_results['all_methods'] == 2).sum(),
    (df_daily_results['all_methods'] == 1).sum()
]
agreement_labels = ['All 3 Methods', 'Exactly 2 Methods', 'Exactly 1 Method']
axes[0, 1].bar(agreement_labels, agreement_counts, color=['red', 'orange', 'gray'], alpha=0.7)
axes[0, 1].set_ylabel('Number of Records')
axes[0, 1].set_title('Anomaly Agreement Between Methods')
for i, v in enumerate(agreement_counts):
    axes[0, 1].text(i, v + 50, str(v), ha='center', fontweight='bold')

# Feature space visualization
scatter = axes[1, 0].scatter(df_daily_results['mean_download_kbps']/1000, 
                              df_daily_results['mean_latency_ms'],
                              c=df_daily_results['all_methods'], 
                              cmap='RdYlGn_r', alpha=0.6)
axes[1, 0].set_xlabel('Download Speed (Mbps)')
axes[1, 0].set_ylabel('Latency (ms)')
axes[1, 0].set_title('Anomaly Score (Number of Methods Flagging)')
plt.colorbar(scatter, ax=axes[1, 0], label='Methods Agreeing')

# Method comparison table
axes[1, 1].axis('off')
table_data = [[m, c, f"{r:.2f}%"] for m, c, r in zip(methods, anomaly_counts, anomaly_rates)]
axes[1, 1].table(cellText=table_data, colLabels=['Method', 'Anomalies', 'Rate'],
                 loc='center', cellLoc='center')
axes[1, 1].set_title('Summary Table')

plt.tight_layout()
plt.savefig('outputs/method_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Answering Project Questions

### Q1. Which regions show the most variable download speed across the 21 days?

In [ ]:
# Q1: Regional variability analysis
regional_stats = df_daily_results.groupby('region').agg({
    'mean_download_kbps': ['mean', 'std', 'count']
})
regional_stats.columns = ['mean_download', 'std_download', 'count']
regional_stats['cv'] = regional_stats['std_download'] / regional_stats['mean_download'] * 100
regional_stats = regional_stats.sort_values('cv', ascending=False).reset_index()

print("Q1: Regions with Most Variable Download Speed")
print("="*60)
print(regional_stats.head(10).to_string(index=False))

In [ ]:
# Visualize Q1
fig, ax = plt.subplots(figsize=(12, 6))
q1_plot = regional_stats.sort_values('cv', ascending=True)
colors = ['red' if cv > q1_plot['cv'].median() else 'steelblue' for cv in q1_plot['cv']]
ax.barh(q1_plot['region'], q1_plot['cv'], color=colors, alpha=0.7)
ax.set_xlabel('Coefficient of Variation (%)')
ax.set_title('Q1: Download Speed Variability by Region')
ax.axvline(x=q1_plot['cv'].median(), color='green', linestyle='--', label='Median CV')
ax.legend()
plt.tight_layout()
plt.savefig('outputs/q1_region_variability.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nQ1 Answer:")
print("-" * 50)
print(f"The regions with the most variable download speeds (highest CV) are:")
for i, row in regional_stats.head(5).iterrows():
    print(f"  {row['region']}: CV = {row['cv']:.1f}%")

### Q2. Which specific days show abnormal network behavior across multiple regions?

In [ ]:
# Q2: Daily anomaly analysis
daily_anomalies = df_daily_results.groupby('aggregate_date').agg({
    'iso_anomaly': 'sum',
    'ocsvm_anomaly': 'sum',
    'ae_anomaly': 'sum',
    'all_methods': 'sum',
    'mean_download_kbps': 'mean',
    'mean_latency_ms': 'mean'
}).reset_index()

# Calculate total unique anomalies (at least one method flagged)
daily_anomalies['total_anomalies'] = daily_anomalies['all_methods']
daily_anomalies = daily_anomalies.sort_values('total_anomalies', ascending=False)

print("Q2: Days with Abnormal Network Behavior")
print("="*60)
print(daily_anomalies[['aggregate_date', 'total_anomalies', 'mean_download_kbps', 'mean_latency_ms']].head(10).to_string(index=False))

In [ ]:
# Visualize Q2
fig, ax = plt.subplots(figsize=(14, 5))
ax.bar(daily_anomalies['aggregate_date'], daily_anomalies['total_anomalies'], color='purple', alpha=0.7)
ax.set_xlabel('Date')
ax.set_ylabel('Number of Anomalies')
ax.set_title('Q2: Daily Anomaly Count Across All Regions')
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('outputs/q2_daily_anomalies.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nQ2 Answer:")
print("-" * 50)
print("Days with the most widespread anomalies (multiple regions affected):")
for i, row in daily_anomalies.head(5).iterrows():
    print(f"  {row['aggregate_date'].strftime('%Y-%m-%d')}: {int(row['total_anomalies'])} anomalies")

### Q3. How consistent is each carrier's performance across regions?

In [ ]:
# Q3: Carrier consistency analysis
carrier_region = df_daily_results.groupby(['carrier_name', 'region'])['mean_download_kbps'].mean().unstack()

print("Q3: Carrier Performance by Region (Download Speed in kbps)")
print("="*60)
print(carrier_region.round(0).to_string())

In [ ]:
# Visualize Q3
fig, ax = plt.subplots(figsize=(14, 8))
sns.heatmap(carrier_region/1000, annot=True, fmt='.1f', cmap='RdYlGn', ax=ax, 
            cbar_kws={'label': 'Download Speed (Mbps)'})
ax.set_title('Q3: Carrier Performance by Region (Download Speed in Mbps)')
plt.tight_layout()
plt.savefig('outputs/q3_carrier_consistency.png', dpi=150, bbox_inches='tight')
plt.show()

# Calculate carrier consistency metrics
carrier_consistency = carrier_region.std(axis=1) / carrier_region.mean(axis=1) * 100
print("\nQ3 Answer:")
print("-" * 50)
print("Carrier Consistency (CV across regions - lower is more consistent):")
for carrier, cv in carrier_consistency.sort_values().items():
    print(f"  {carrier}: CV = {cv:.1f}%")

### Q4. How does LTE compare to 5G in terms of anomaly patterns?

In [ ]:
# Q4: LTE vs 5G comparison
tech_comparison = df_daily_results.groupby('technology_type').agg({
    'iso_anomaly': ['sum', 'mean'],
    'ocsvm_anomaly': ['sum', 'mean'],
    'ae_anomaly': ['sum', 'mean'],
    'mean_download_kbps': 'mean',
    'mean_latency_ms': 'mean'
})
tech_comparison.columns = ['_'.join(col) for col in tech_comparison.columns]
tech_comparison = tech_comparison.reset_index()

print("Q4: LTE vs 5G Comparison")
print("="*60)
print(tech_comparison.to_string(index=False))

In [ ]:
# Visualize Q4
df_lte = df_daily_results[df_daily_results['technology_type'] == 'LTE']
df_5g = df_daily_results[df_daily_results['technology_type'] == '5G']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Anomaly rates
techs = ['LTE', '5G']
iso_rates = [df_lte['iso_anomaly'].mean()*100, df_5g['iso_anomaly'].mean()*100]
ae_rates = [df_lte['ae_anomaly'].mean()*100, df_5g['ae_anomaly'].mean()*100]

x = np.arange(len(techs))
width = 0.35

axes[0].bar(x - width/2, iso_rates, width, label='Isolation Forest', color='blue', alpha=0.7)
axes[0].bar(x + width/2, ae_rates, width, label='Autoencoder', color='green', alpha=0.7)
axes[0].set_ylabel('Anomaly Rate (%)')
axes[0].set_title('Q4: Anomaly Rates by Technology')
axes[0].set_xticks(x)
axes[0].set_xticklabels(techs)
axes[0].legend()

# Performance comparison
axes[1].scatter(df_lte['mean_download_kbps']/1000, df_lte['mean_latency_ms'], 
                alpha=0.5, label='LTE', c='blue')
axes[1].scatter(df_5g['mean_download_kbps']/1000, df_5g['mean_latency_ms'], 
                alpha=0.5, label='5G', c='green')
axes[1].set_xlabel('Download Speed (Mbps)')
axes[1].set_ylabel('Latency (ms)')
axes[1].set_title('Q4: Performance by Technology')
axes[1].legend()

plt.tight_layout()
plt.savefig('outputs/q4_lte_vs_5g.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nQ4 Answer:")
print("-" * 50)
print(f"LTE: {len(df_lte)} records, {df_lte['iso_anomaly'].sum()} anomalies ({df_lte['iso_anomaly'].mean()*100:.1f}%)")
print(f"5G: {len(df_5g)} records, {df_5g['iso_anomaly'].sum()} anomalies ({df_5g['iso_anomaly'].mean()*100:.1f}%)")

### Q5. What are the top 5 worst performing combinations of region, carrier, and technology?

In [ ]:
# Q5: Top 5 worst performers
# Calculate worst score
df_daily_results['download_score'] = (df_daily_results['mean_download_kbps'].max() - df_daily_results['mean_download_kbps']) / df_daily_results['mean_download_kbps'].max()
df_daily_results['upload_score'] = (df_daily_results['mean_upload_kbps'].max() - df_daily_results['mean_upload_kbps']) / df_daily_results['mean_upload_kbps'].max()
df_daily_results['latency_score'] = df_daily_results['mean_latency_ms'] / df_daily_results['mean_latency_ms'].max()

df_daily_results['worst_score'] = (
    0.4 * df_daily_results['download_score'] +
    0.2 * df_daily_results['upload_score'] +
    0.3 * df_daily_results['latency_score'] +
    0.1 * df_daily_results['all_methods'] / 3
)

top5_worst = df_daily_results.nlargest(5, 'worst_score')

print("Q5: Top 5 Worst Performing Combinations")
print("="*60)
for i, (_, row) in enumerate(top5_worst.iterrows(), 1):
    print(f"\n{i}. {row['region']} - {row['carrier_name']} - {row['technology_type']}")
    print(f"   Date: {row['aggregate_date'].strftime('%Y-%m-%d')}")
    print(f"   Download: {row['mean_download_kbps']/1000:.2f} Mbps")
    print(f"   Upload: {row['mean_upload_kbps']/1000:.2f} Mbps")
    print(f"   Latency: {row['mean_latency_ms']:.1f} ms")
    print(f"   Worst Score: {row['worst_score']:.4f}")
    print(f"   Anomaly Flags: {int(row['all_methods'])} methods")

In [ ]:
# Visualize Q5
fig, ax = plt.subplots(figsize=(10, 5))
labels = [f"{row['region']}\n{row['carrier_name']}\n{row['aggregate_date'].strftime('%m/%d')}" 
          for _, row in top5_worst.iterrows()]
ax.barh(range(5), top5_worst['worst_score'].values, color='red', alpha=0.7)
ax.set_yticks(range(5))
ax.set_yticklabels(labels)
ax.set_xlabel('Worst Score')
ax.set_title('Q5: Top 5 Worst Performing Combinations')
ax.invert_yaxis()
plt.tight_layout()
plt.savefig('outputs/q5_top5_worst.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Save Results

In [ ]:
# Create anomaly log
anomaly_log = df_daily_results[
    (df_daily_results['iso_anomaly'] == 1) | 
    (df_daily_results['ocsvm_anomaly'] == 1) | 
    (df_daily_results['ae_anomaly'] == 1)
].copy()

def get_detection_methods(row):
    methods = []
    if row['iso_anomaly'] == 1:
        methods.append('IsolationForest')
    if row['ocsvm_anomaly'] == 1:
        methods.append('OC-SVM')
    if row['ae_anomaly'] == 1:
        methods.append('Autoencoder')
    return ', '.join(methods)

anomaly_log['detection_methods'] = anomaly_log.apply(get_detection_methods, axis=1)
anomaly_log['methods_count'] = anomaly_log['iso_anomaly'] + anomaly_log['ocsvm_anomaly'] + anomaly_log['ae_anomaly']
anomaly_log = anomaly_log.sort_values(['methods_count', 'aggregate_date'], ascending=[False, True])

# Save outputs
import os
os.makedirs('outputs', exist_ok=True)

df_daily_results.to_csv('processed_data_with_anomalies.csv', index=False)
anomaly_log.to_csv('anomaly_log.csv', index=False)

print(f"Saved {len(df_daily_results)} records to processed_data_with_anomalies.csv")
print(f"Saved {len(anomaly_log)} anomalies to anomaly_log.csv")

In [ ]:
# Final Summary
print("\n" + "="*60)
print("ANALYSIS COMPLETE - SUMMARY")
print("="*60)
print(f"\nTotal records analyzed: {len(df_daily_results):,}")
print(f"Total anomalies detected: {len(anomaly_log):,}")
print(f"High confidence anomalies (3 methods): {(df_daily_results['all_methods'] == 3).sum()}")
print(f"\nAnomaly Detection Methods:")
print(f"  - Isolation Forest: {df_daily_results['iso_anomaly'].sum()} anomalies")
print(f"  - One-Class SVM: {df_daily_results['ocsvm_anomaly'].sum()} anomalies")
print(f"  - PyTorch Autoencoder: {df_daily_results['ae_anomaly'].sum()} anomalies")
print(f"\nAll outputs saved to the outputs/ directory")